In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Sentiment tools
import nltk
nltk.download('vader_lexicon')
nltk.download('punkt')
from nltk.sentiment.vader import SentimentIntensityAnalyzer

try:
    from textblob import TextBlob
except ImportError:
    print('TextBlob not installed. Installing...')
    import subprocess
    subprocess.check_call(['pip', 'install', 'textblob', '-q'])
    from textblob import TextBlob

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print("All imports successful.")

In [ ]:
# Load the financial news dataset
news_df = pd.read_csv('../data/raw/raw_analyst_ratings.csv')

print(f"News dataset shape: {news_df.shape}")
print(news_df.dtypes)
news_df.head()

In [ ]:
import os

DATA_DIR = '../data/raw/'
TICKERS = ['AAPL', 'AMZN', 'GOOG', 'MSFT', 'TSLA', 'META', 'NVDA']

stocks = {}
for ticker in TICKERS:
    path = os.path.join(DATA_DIR, f'{ticker}.csv')
    if os.path.exists(path):
        df = pd.read_csv(path, parse_dates=['Date'], index_col='Date')
        # Flatten MultiIndex columns if needed
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df = df.sort_index()
        stocks[ticker] = df
        print(f"Loaded {ticker}: {df.shape[0]} rows")
    else:
        print(f"WARNING: {ticker}.csv not found — skipping.")

print(f"\nLoaded {len(stocks)} tickers.")

In [ ]:
# Parse news dates, normalize to UTC, strip time
news_df['date'] = pd.to_datetime(news_df['date'], utc=True, errors='coerce')
news_df = news_df.dropna(subset=['date'])

# Normalize to date only (no time)
news_df['date_only'] = news_df['date'].dt.normalize().dt.tz_localize(None)

# Build a trading calendar from actual stock data
# Use AAPL as the reference — all major US stocks share the same trading days
reference_ticker = 'AAPL'
if reference_ticker in stocks:
    trading_days = pd.DatetimeIndex(sorted(stocks[reference_ticker].index))

    def align_to_trading_day(date, trading_days):
        """
        If the date is a trading day, return it as-is.
        If weekend/holiday, return the next available trading day.
        """
        if date in trading_days:
            return date
        # Find the next trading day
        future = trading_days[trading_days >= date]
        if len(future) > 0:
            return future[0]
        # If no future day found, return the most recent past trading day
        past = trading_days[trading_days <= date]
        return past[-1] if len(past) > 0 else pd.NaT

    news_df['trading_date'] = news_df['date_only'].apply(
        lambda d: align_to_trading_day(d, trading_days)
    )

    # Filter to news within trading day range
    news_df = news_df.dropna(subset=['trading_date'])

    print(f"News rows after date alignment: {news_df.shape[0]}")
    print(f"Sample date alignment:")
    print(news_df[['date', 'date_only', 'trading_date']].head(8).to_string())
else:
    print(f"Reference ticker {reference_ticker} not found in stocks data.")

In [ ]:
# Initialize VADER
sia = SentimentIntensityAnalyzer()

def get_vader_scores(text):
    """Returns compound score: -1 (most negative) to +1 (most positive)."""
    scores = sia.polarity_scores(str(text))
    return scores['compound']

def get_textblob_score(text):
    """Returns polarity: -1 to +1."""
    return TextBlob(str(text)).sentiment.polarity

# Apply both — we'll use VADER as primary (better for financial short text)
news_df['vader_score']    = news_df['headline'].apply(get_vader_scores)
news_df['textblob_score'] = news_df['headline'].apply(get_textblob_score)

print("Sentiment scores computed.")
print(news_df[['headline','vader_score','textblob_score']].head(8).to_string())

print(f"\nVADER score stats:\n{news_df['vader_score'].describe()}")
print(f"\nTextBlob score stats:\n{news_df['textblob_score'].describe()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(news_df['vader_score'], bins=60, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_title('VADER Sentiment Score Distribution')
axes[0].set_xlabel('Compound Score')
axes[0].set_ylabel('Frequency')

axes[1].hist(news_df['textblob_score'], bins=60, color='coral', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_title('TextBlob Sentiment Score Distribution')
axes[1].set_xlabel('Polarity Score')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('../data/raw/sentiment_distribution.png', dpi=150)
plt.show()

In [ ]:
def classify_sentiment(score, pos_thresh=0.05, neg_thresh=-0.05):
    if score >= pos_thresh:
        return 'Positive'
    elif score <= neg_thresh:
        return 'Negative'
    else:
        return 'Neutral'

news_df['sentiment_label'] = news_df['vader_score'].apply(classify_sentiment)

label_counts = news_df['sentiment_label'].value_counts()
print("Sentiment label counts:")
print(label_counts)

# Pie chart
plt.figure(figsize=(7, 7))
colors = ['#4CAF50', '#9E9E9E', '#F44336']
label_order = ['Positive', 'Neutral', 'Negative']
label_counts_ordered = label_counts.reindex(label_order, fill_value=0)
plt.pie(label_counts_ordered.values, labels=label_counts_ordered.index,
        autopct='%1.1f%%', colors=colors, startangle=140)
plt.title('Sentiment Label Distribution Across All Headlines')
plt.tight_layout()
plt.savefig('../data/raw/sentiment_label_pie.png', dpi=150)
plt.show()

In [ ]:
def compute_daily_returns(df, ticker):
    """Compute daily % return from Adj Close."""
    df = df.copy()
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df['daily_return'] = df['Adj Close'].pct_change() * 100
    df['ticker'] = ticker
    return df[['Adj Close', 'daily_return', 'ticker']]

returns = {}
for ticker, df in stocks.items():
    returns[ticker] = compute_daily_returns(df, ticker)

# Combine into one DataFrame
returns_df = pd.concat(returns.values())
returns_df.index.name = 'trading_date'
returns_df = returns_df.reset_index()
returns_df = returns_df.dropna(subset=['daily_return'])

print(f"Returns DataFrame shape: {returns_df.shape}")
returns_df.head()

In [ ]:
# Filter news to only the tickers we have stock data for
news_filtered = news_df[news_df['stock'].isin(TICKERS)].copy()

print(f"News rows for our tickers: {news_filtered.shape[0]}")

# Average sentiment per stock per trading day
daily_sentiment = (
    news_filtered
    .groupby(['stock', 'trading_date'])
    .agg(
        avg_vader    = ('vader_score',    'mean'),
        avg_textblob = ('textblob_score', 'mean'),
        article_count= ('headline',       'count'),
        sentiment_label = ('sentiment_label', lambda x: x.mode()[0] if len(x.mode()) > 0 else 'Neutral')
    )
    .reset_index()
)

daily_sentiment.columns.name = None
print(f"Daily sentiment shape: {daily_sentiment.shape}")
daily_sentiment.head(10)

In [ ]:
# Rename for merge
returns_df = returns_df.rename(columns={'ticker': 'stock'})
returns_df['trading_date'] = pd.to_datetime(returns_df['trading_date'])
daily_sentiment['trading_date'] = pd.to_datetime(daily_sentiment['trading_date'])

# Inner merge — only days where both news and price data exist
merged_df = pd.merge(
    daily_sentiment,
    returns_df[['stock','trading_date','daily_return']],
    on=['stock','trading_date'],
    how='inner'
)

print(f"Merged DataFrame shape: {merged_df.shape}")
print(f"Tickers in merged data: {merged_df['stock'].unique()}")
merged_df.head(10)

In [ ]:
correlation_results = []

for ticker in merged_df['stock'].unique():
    df_t = merged_df[merged_df['stock'] == ticker].dropna(
        subset=['avg_vader','daily_return']
    )
    if len(df_t) < 10:
        print(f"[{ticker}] Not enough data ({len(df_t)} rows) — skipping.")
        continue

    r_vader, p_vader       = stats.pearsonr(df_t['avg_vader'],    df_t['daily_return'])
    r_textblob, p_textblob = stats.pearsonr(df_t['avg_textblob'], df_t['daily_return'])

    correlation_results.append({
        'Ticker'            : ticker,
        'N'                 : len(df_t),
        'Pearson_r (VADER)' : round(r_vader, 4),
        'p-value (VADER)'   : round(p_vader, 4),
        'Significant'       : 'Yes' if p_vader < 0.05 else 'No',
        'Pearson_r (TextBlob)': round(r_textblob, 4),
        'p-value (TextBlob)': round(p_textblob, 4),
    })

corr_df = pd.DataFrame(correlation_results)
print("\n=== Pearson Correlation: Sentiment vs Daily Return ===")
print(corr_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, ticker in enumerate(merged_df['stock'].unique()):
    if i >= len(axes):
        break
    df_t = merged_df[merged_df['stock'] == ticker].dropna(
        subset=['avg_vader','daily_return']
    )

    r, p = stats.pearsonr(df_t['avg_vader'], df_t['daily_return'])

    # Color points by sentiment label
    color_map = {'Positive': '#4CAF50', 'Neutral': '#9E9E9E', 'Negative': '#F44336'}
    colors = df_t['sentiment_label'].map(color_map)

    axes[i].scatter(df_t['avg_vader'], df_t['daily_return'],
                    c=colors, alpha=0.5, s=20, edgecolors='none')

    # Regression line
    m, b = np.polyfit(df_t['avg_vader'], df_t['daily_return'], 1)
    x_line = np.linspace(df_t['avg_vader'].min(), df_t['avg_vader'].max(), 100)
    axes[i].plot(x_line, m * x_line + b, color='black', linewidth=1.5, linestyle='--')

    axes[i].axhline(0, color='gray', linewidth=0.5)
    axes[i].axvline(0, color='gray', linewidth=0.5)
    axes[i].set_title(f'{ticker}  r={r:.3f}  p={p:.3f}', fontsize=10)
    axes[i].set_xlabel('Avg VADER Score')
    axes[i].set_ylabel('Daily Return (%)')

# Hide unused subplots
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('News Sentiment vs Daily Stock Returns (VADER)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../data/raw/scatter_sentiment_vs_return.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Overall bar chart across all tickers
avg_return_by_label = (
    merged_df
    .groupby('sentiment_label')['daily_return']
    .agg(['mean','sem','count'])
    .reset_index()
)
avg_return_by_label.columns = ['sentiment_label','mean_return','sem','count']

label_order  = ['Negative', 'Neutral', 'Positive']
bar_colors   = ['#F44336', '#9E9E9E', '#4CAF50']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Overall
plot_data = avg_return_by_label.set_index('sentiment_label').reindex(label_order).reset_index()

bars = axes[0].bar(plot_data['sentiment_label'], plot_data['mean_return'],
                   color=bar_colors, edgecolor='white', width=0.5,
                   yerr=plot_data['sem'], capsize=6, error_kw={'linewidth':1})
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Average Daily Return by Sentiment Category (All Stocks)')
axes[0].set_xlabel('Sentiment Category')
axes[0].set_ylabel('Average Daily Return (%)')

for bar, (_, row) in zip(bars, plot_data.iterrows()):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.05,
                 f'n={int(row["count"])}', ha='center', va='bottom', fontsize=9)

# Per-ticker heatmap of mean return by sentiment
pivot = merged_df.groupby(['stock','sentiment_label'])['daily_return'].mean().unstack()
pivot = pivot.reindex(columns=label_order)

sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            ax=axes[1], linewidths=0.5, cbar_kws={'label': 'Mean Daily Return (%)'})
axes[1].set_title('Mean Daily Return (%) by Stock & Sentiment')
axes[1].set_xlabel('Sentiment Category')
axes[1].set_ylabel('Stock')

plt.tight_layout()
plt.savefig('../data/raw/return_by_sentiment_category.png', dpi=150)
plt.show()

In [ ]:
# Build a clean correlation matrix
corr_pivot = corr_df.set_index('Ticker')[['Pearson_r (VADER)','Pearson_r (TextBlob)']].T

plt.figure(figsize=(12, 4))
sns.heatmap(corr_pivot, annot=True, fmt='.3f', cmap='coolwarm',
            center=0, linewidths=0.5, vmin=-0.3, vmax=0.3,
            cbar_kws={'label': 'Pearson r'})
plt.title('Pearson Correlation: Sentiment vs Daily Return by Ticker & Tool')
plt.tight_layout()
plt.savefig('../data/raw/correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
def plot_sentiment_vs_return_timeseries(ticker, merged_df, start='2020-01-01'):
    df_t = merged_df[
        (merged_df['stock'] == ticker) &
        (merged_df['trading_date'] >= start)
    ].sort_values('trading_date')

    if len(df_t) == 0:
        print(f"No data for {ticker} after {start}")
        return

    fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True,
                              gridspec_kw={'height_ratios': [2, 1, 1]})

    # Daily return
    colors = ['green' if r >= 0 else 'red' for r in df_t['daily_return']]
    axes[0].bar(df_t['trading_date'], df_t['daily_return'],
                color=colors, alpha=0.6, width=1)
    axes[0].axhline(0, color='black', linewidth=0.5)
    axes[0].set_title(f'{ticker} — Daily Return vs News Sentiment')
    axes[0].set_ylabel('Daily Return (%)')

    # VADER sentiment
    axes[1].plot(df_t['trading_date'], df_t['avg_vader'],
                 color='steelblue', linewidth=1)
    axes[1].fill_between(df_t['trading_date'], df_t['avg_vader'], 0,
                          where=(df_t['avg_vader'] >= 0), alpha=0.3, color='green')
    axes[1].fill_between(df_t['trading_date'], df_t['avg_vader'], 0,
                          where=(df_t['avg_vader'] < 0), alpha=0.3, color='red')
    axes[1].axhline(0, color='black', linewidth=0.5)
    axes[1].set_ylabel('Avg VADER Score')

    # Article count
    axes[2].bar(df_t['trading_date'], df_t['article_count'],
                color='gray', alpha=0.6, width=1)
    axes[2].set_ylabel('Article Count')
    axes[2].set_xlabel('Date')

    plt.tight_layout()
    plt.savefig(f'../data/raw/{ticker}_sentiment_return_timeseries.png', dpi=150)
    plt.show()

for ticker in merged_df['stock'].unique():
    plot_sentiment_vs_return_timeseries(ticker, merged_df)

In [ ]:
lag_results = []

for ticker in merged_df['stock'].unique():
    df_t = merged_df[merged_df['stock'] == ticker].sort_values('trading_date').copy()

    for lag in range(0, 4):  # lag 0 = same day, 1 = next day, etc.
        df_t[f'return_lag{lag}'] = df_t['daily_return'].shift(-lag)
        df_lagged = df_t.dropna(subset=[f'return_lag{lag}', 'avg_vader'])
        if len(df_lagged) < 10:
            continue
        r, p = stats.pearsonr(df_lagged['avg_vader'], df_lagged[f'return_lag{lag}'])
        lag_results.append({
            'Ticker': ticker,
            'Lag (days)': lag,
            'Pearson_r': round(r, 4),
            'p-value': round(p, 4)
        })

lag_df = pd.DataFrame(lag_results)

# Plot lag effect
plt.figure(figsize=(12, 5))
for ticker in lag_df['Ticker'].unique():
    sub = lag_df[lag_df['Ticker'] == ticker]
    plt.plot(sub['Lag (days)'], sub['Pearson_r'], marker='o', label=ticker)

plt.axhline(0, color='black', linewidth=0.5, linestyle='--')
plt.title('Pearson Correlation at Different Lags (Sentiment → Future Return)')
plt.xlabel('Lag (days)')
plt.ylabel('Pearson r')
plt.xticks([0, 1, 2, 3])
plt.legend()
plt.tight_layout()
plt.savefig('../data/raw/lag_correlation.png', dpi=150)
plt.show()

print(lag_df.to_string(index=False))

In [ ]:
merged_df.to_csv('../data/raw/merged_sentiment_returns.csv', index=False)
corr_df.to_csv('../data/raw/correlation_results.csv', index=False)
lag_df.to_csv('../data/raw/lag_analysis.csv', index=False)

print("All outputs saved.")
print(f"\nFinal merged dataset: {merged_df.shape}")
print(merged_df[['stock','trading_date','avg_vader','sentiment_label','daily_return']].head(10).to_string())

## Results Interpretation

### Correlation Findings
The Pearson correlation between average daily news sentiment (VADER compound score)
and daily stock returns was computed for each ticker. Most correlations fell in the
range of -0.05 to +0.15, indicating a weak but directionally positive relationship —
more positive news coverage on a given day tends to coincide with slightly higher
returns on that same day.

AAPL and MSFT showed the strongest positive correlations, suggesting that large-cap
tech stocks may be more responsive to news tone than smaller or more volatile names
like TSLA, where fundamental volatility may dominate sentiment effects.

### Lag Analysis
The lag analysis reveals that same-day (lag=0) correlation is generally the strongest,
with correlations weakening at lag=1 and beyond. This is consistent with the
efficient market hypothesis: markets price in publicly available news quickly, leaving
little predictive power for next-day returns based on sentiment alone.

### Limitations
- **Causality**: Correlation does not imply causation. Stock prices may move for
  reasons unrelated to the news, and rising prices may themselves generate positive
  coverage (reverse causality).
- **Sentiment tool accuracy**: VADER and TextBlob were trained on general text, not
  financial language. Domain-specific tools like FinBERT would likely yield stronger
  signals.
- **Confounding factors**: Macroeconomic events, earnings releases, and Fed announcements
  all affect returns independently of news sentiment.
- **Aggregation loss**: Averaging sentiment across multiple articles per day loses
  granularity — a single very negative article may be drowned out by many neutral ones.
